<a href="https://colab.research.google.com/github/stefisha/NordeusChallenge/blob/improved/NordeusChallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Nordeus Data Science Challenge (for JobFair 2023)
Submission for the challenge explained [here](https://nordeus.com/data-science-challenge/).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
# Load the datasets
train_data = pd.read_csv('/content/drive/MyDrive/Nordeus Challenge/jobfair_train.csv')
test_data = pd.read_csv('/content/drive/MyDrive/Nordeus Challenge/jobfair_test.csv')

In [ ]:
test_data.head()

In [ ]:
# Identify categorical and numerical columns
categorical_cols = train_data.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = train_data.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [ ]:
# Remove the target variable and identifiers from the numerical columns list
numerical_cols.remove('league_rank')
numerical_cols.remove('club_id')
numerical_cols.remove('league_id')

In [ ]:
# Preprocessing for numerical data
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
    #('scaler', StandardScaler()) # maybe unnecessary
])

In [ ]:
# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [ ]:
# Separate target from predictors
X_train = train_data.drop(['club_id', 'league_id', 'league_rank'], axis=1)
y_train = train_data['league_rank']
X_test = test_data.drop(['club_id', 'league_id'], axis=1)

In [ ]:
# Preprocess the data
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

In [ ]:
# Convert the preprocessed training data to a DataFrame
ohe_feature_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_cols)
feature_names = list(numerical_cols) + list(ohe_feature_names)
X_train_preprocessed_df = pd.DataFrame(X_train_preprocessed.toarray(), columns=feature_names)

In [ ]:
# Define the model
model = RandomForestClassifier(random_state=0, n_estimators=100)

In [ ]:
# Perform cross-validation
cv_scores = -1 * cross_val_score(model, X_train_preprocessed_df, y_train, cv=5, scoring='neg_mean_absolute_error')

In [ ]:
# Print the mean and standard deviation of the cross-validation scores
print(f'Mean MAE: {cv_scores.mean()}')
print(f'Standard Deviation of MAE: {cv_scores.std()}')

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Define the parameter distribution to sample from
param_dist = {
#     'n_estimators': randint(50, 200),
#     'max_depth': [None, 10, 20, 30],
    'n_estimators': randint(50, 100),  # Reduced upper limit
    'max_depth': [None, 5, 10, 15],  # Lower depths
    'min_samples_split': randint(2, 11),
    'min_samples_leaf': randint(1, 5)
}

# Define the model
rf = RandomForestClassifier(random_state=0)

# Define RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_dist,
                                   n_iter=10, cv=3, n_jobs=-1, ## instead of 100,3
                                   scoring='neg_mean_absolute_error', verbose=3)

# Fit the random search to the data
random_search.fit(X_train_preprocessed_df, y_train)

# Print the best parameters and best score
print("Best Parameters:", random_search.best_params_)
print("Best MAE:", -random_search.best_score_)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Define a simple Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=0)

# Perform cross-validation to evaluate the model using Mean Absolute Error
# We multiply by -1 because sklearn's cross-validation functions maximize utility functions, so they treat MAE as negative
dt_cv_scores = -1 * cross_val_score(dt_model, X_train_preprocessed, y_train, cv=5, scoring='neg_mean_absolute_error')

# Compute the mean and standard deviation of the cross-validation scores
dt_cv_mae_mean = dt_cv_scores.mean()
dt_cv_mae_std = dt_cv_scores.std()

dt_cv_mae_mean, dt_cv_mae_std

In [ ]:
# Adjust the target variable to start from 0
y_train_adjusted = y_train - 1

# Define the XGBoost model
xgb_model = xgb.XGBClassifier(objective='multi:softprob', random_state=0, num_class=14)

# Perform cross-validation
xgb_cv_scores = -1 * cross_val_score(xgb_model, X_train_preprocessed, y_train_adjusted, cv=5, scoring='neg_mean_absolute_error')

# Compute the mean and standard deviation of the cross-validation scores
xgb_cv_mae_mean = xgb_cv_scores.mean()
xgb_cv_mae_std = xgb_cv_scores.std()

print(f"XGBoost Mean MAE: {xgb_cv_mae_mean}")
print(f"XGBoost Standard Deviation of MAE: {xgb_cv_mae_std}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

# Hyperparameter grid
param_dist = {
    'n_estimators': randint(50, 500),
    'learning_rate': uniform(0.01, 0.6),
    'max_depth': randint(3, 10),
    'min_child_weight': randint(1, 10),
    'gamma': uniform(0, 0.5),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5)
}
# param_dist = { ## computationally too demanding
#     'n_estimators': randint(100, 1000),  # Wider range
#     'learning_rate': uniform(0.01, 0.1),  # Narrower range for more precision
#     'max_depth': randint(3, 15),  # Try deeper trees
#     'min_child_weight': randint(1, 6),
#     'gamma': uniform(0, 0.5),
#     'subsample': uniform(0.5, 0.5),
#     'colsample_bytree': uniform(0.5, 0.5)
# }


# Initialize XGBoost model
#xgb_model = xgb.XGBClassifier(objective='multi:softprob', random_state=0, num_class=14)
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    random_state=0,
    # num_class=14,
    #early_stopping_rounds=10,
    eval_metric="mae",
    verbose=True
)

# RandomizedSearchCV
random_search = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_dist,
    n_iter=10,
    scoring='neg_mean_absolute_error',
    cv=5,
    random_state=0,
    verbose=3,
    n_jobs=-1
)

# Fit the model
random_search.fit(X_train_preprocessed, y_train_adjusted)

# Print the best parameters and lowest MAE
print("Best parameters found: ", random_search.best_params_)
print("Lowest MAE found: ", -random_search.best_score_)

In [ ]:
# Train the final model with the best parameters
final_xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    colsample_bytree=0.6044,
    gamma=0.0807,
    learning_rate=0.4019,
    max_depth=3,
    min_child_weight=5,
    n_estimators=277,
    subsample=0.8118,
    random_state=0,
    num_class=14
)

final_xgb_model.fit(X_train_preprocessed, y_train_adjusted)

In [ ]:
# Train the final model with the best parameters
final_xgb_model.fit(X_train_preprocessed, y_train_adjusted)

# Get feature importance
feature_importances = final_xgb_model.feature_importances_

# Sort features by their importance scores
sorted_idx = feature_importances.argsort()

# Select the top N features (e.g., top 20)
top_n = 20
sorted_idx_top = sorted_idx[-top_n:]

plt.figure(figsize=(10, 8))
plt.barh(range(len(sorted_idx_top)), feature_importances[sorted_idx_top], align='center')
plt.yticks(range(len(sorted_idx_top)), [feature_names[i] for i in sorted_idx_top])
plt.xlabel('Feature Importance')
plt.ylabel('Feature')
plt.show()


In [ ]:
# # Make predictions
# predictions = final_xgb_model.predict(X_test_preprocessed)

# # Adjust predictions back to original scale (because we subtracted 1 earlier)
# predictions_adjusted = predictions + 1

In [ ]:
# # Create a DataFrame for submission
# submission = pd.DataFrame({
#     'club_id': test_data['club_id'],
#     'predicted_league_rank': predictions_adjusted
# })

# # Display the first few rows of the submission DataFrame
# submission.head()

In [ ]:
# # Save the DataFrame to a CSV file for submission
# submission.to_csv('league_rank_predictions.csv', index=False)

In [ ]:
import lightgbm as lgb
lgb_model = lgb.LGBMClassifier(objective='multiclass', num_class=14, random_state=0)
lgb_cv_scores = -1 * cross_val_score(lgb_model, X_train_preprocessed, y_train_adjusted, cv=5, scoring='neg_mean_absolute_error')

In [ ]:
lgb_cv_mae_mean = lgb_cv_scores.mean()
lgb_cv_mae_std = lgb_cv_scores.std()

print(f"LGBoost Mean MAE: {lgb_cv_mae_mean}")
print(f"LGBoost Standard Deviation of MAE: {lgb_cv_mae_std}")

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('xgb', xgb.XGBClassifier(objective='multi:softprob', num_class=14, random_state=0))
]
stack_model = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(max_iter=1000))

In [ ]:
from sklearn.model_selection import StratifiedKFold
stratified_cv = StratifiedKFold(n_splits=5)
scores = -1 * cross_val_score(stack_model, X_train_preprocessed, y_train_adjusted, cv=stratified_cv, scoring='neg_mean_absolute_error')

In [ ]:
# Compute the mean and standard deviation of the cross-validation scores
stacked_mean = scores.mean()
stacked_mae_std = scores.std()

print(f"Stacking Mean MAE: {stacked_mean}")
print(f"Stacking Standard Deviation of MAE: {stacked_mae_std}")

In [ ]:
# Fit data to final model
stack_model.fit(X_train_preprocessed, y_train_adjusted)
# Make predictions
final_predictions = stack_model.predict(X_test_preprocessed)

# Adjust predictions back to original scale (because we subtracted 1 earlier)
predictions_adjusted = final_predictions + 1

In [ ]:
# Create a DataFrame for submission
submission = pd.DataFrame({
    'club_id': test_data['club_id'],
    'predicted_league_rank': predictions_adjusted
})

# Display the first few rows of the submission DataFrame
submission.head()

In [ ]:
# Save the DataFrame to a CSV file for submission
submission.to_csv('league_rank_predictions.csv', index=False)